## Import

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
import duckdb
import pandas as pd

from sindex.metrics.citations import merge_citations_from_files_fast

from sindex.metrics.mentions import merge_mentions_from_files_fast

from sindex.metrics.fairscores import merge_doi_fair_scores_ndjson_files, extrapolate_emdb_fair_scores
from sindex.utils.files import combine_ndjson_files, merge_ndjson_files_in_folder
from sindex.metrics.topics import enhance_topics, restructure_topics_ndjson
from sindex.metrics.batch_jobs import (
    batch_process_metadata_from_slim, 
    create_metadata_table,
    create_citations_table,
    create_mentions_table,
    create_fair_scores_table,
    create_topics_table,
    create_dataset_metrics_table,
    calculate_normalization_factors_topics,
    calculate_normalization_factors_subfields,
    calculate_normalization_factors,
    create_floored_normalization_factors_table,
    create_dataset_index_table,
    create_creators_table,
    create_s_index_identifier_table,
    create_s_index_name_affiliation_table,
    create_s_index_identifier_name_affiliation_table,
)

### Deduplicate citations from different sources and combines

In [4]:
mdc_citations = r"D:\pipeline-data\citations\mdc\mdc_citations_datacite.ndjson"
oa_citations = r"D:\pipeline-data\citations\openalex\oa_citations.ndjson"
dc_citations = r"D:\pipeline-data\citations\datacite\dc_citations.ndjson"
mdc_citations_emdb = r"D:\pipeline-data\citations\mdc\mdc_citations_emdb.ndjson"
citation_files = [mdc_citations, oa_citations, dc_citations, mdc_citations_emdb]
output_file = r"D:\pipeline-data\citations\citations.ndjson"

In [5]:
merge_citations_from_files_fast(citation_files, output_file)

Starting merge of 4 valid files...
Finished processing 8,876,816 records. Unique: 7,669,263
Writing to D:\pipeline-data\citations\citations.ndjson...
Done!


## Mentions Preprocessing

### Deduplicate mentions from different sources and combines

In [7]:
hf_mentions_mc = r"D:\pipeline-data\mentions\huggingface\hf_mentions_modelcard.ndjson"
swh_mentions_datacite = r"D:\pipeline-data\mentions\swh\swh_mentions_datacite.ndjson"
swh_mentions_emdb = r"D:\pipeline-data\mentions\swh\swh_mentions_emdb.ndjson"
uspto_mentions_emdb = r"D:\pipeline-data\mentions\uspto\uspto_mentions.ndjson"
mention_files = [hf_mentions_mc, swh_mentions_datacite, swh_mentions_emdb, uspto_mentions_emdb]
output_file = r"D:\pipeline-data\mentions\mentions.ndjson"

In [8]:
merge_mentions_from_files_fast(mention_files, output_file)

Starting merge of 4 valid files...
Finished processing 91,906 records. Unique: 91,891
Writing to D:\pipeline-data\mentions\mentions.ndjson...
Done!


## FAIR scores preprocessing

### Merge DOI FAIR score into one file (rename doi field as dataset_id)

In [15]:
fair_scores_directory = r"D:\pipeline-data\fair_scores\fair_scores_doi_files"
doi_fair_scores_path = r"D:\pipeline-data\fair_scores\doi_fair_scores.ndjson"

In [16]:
merge_doi_fair_scores_ndjson_files(fair_scores_directory, doi_fair_scores_path)

Found 4901 files. Starting merge with orjson...
Done! Total lines in 'D:\pipeline-data\fair_scores\doi_fair_scores.ndjson': 49,009,521    


In [17]:
#Add missing 1 DOI (based on F-UJI website score)
new_entry = {
    "dataset_id": "10.57451/lhd.ficxs-2.156223.1",
    "score": 65.00,
    "evaluationDate": "2026-02-02T00:00:00+00:00",
    "metricVersion": "0.8",
    "softwareVersion": "website"
}

file_path = doi_fair_scores_path

with open(file_path, 'a') as f:
    f.write(json.dumps(new_entry) + '\n')

### Extrapolate EMDB fair scores (all the same on the first 10k calculated with F-UJI)

In [20]:
emdb_file_path = r"D:\pipeline-data\records\slim-records\emdb-slim-records\emdb-records-slim.ndjson"
partial_score_file_path = r"D:\pipeline-data\fair_scores\fair_scores_emdb_partial\fair_scores_emdb_partial.ndjson"
emdb_fair_scores_path = r"D:\pipeline-data\fair_scores\emdb_fair_scores.ndjson"

In [30]:
extrapolate_emdb_fair_scores(emdb_file_path, partial_score_file_path, emdb_fair_scores_path)

Loading scores from D:\pipeline-data\fair_scores\fair_scores_emdb_partial\fair_scores_emdb_partial.ndjson...
Processing D:\pipeline-data\records\slim-records\emdb-slim-records\emdb-records-slim.ndjson...
----------------------------------------
STATISTICS (orjson)
----------------------------------------
Total records in EMDB file:   51645
Total records written:        51645
  - Found existing scores:    18132
  - Extrapolated scores:      33513
  - Skipped (no ID):          0
----------------------------------------
SUCCESS: Input count matches output count.


### Merge all FAIR score into one file

In [21]:
fair_scores_path = r"D:\pipeline-data\fair_scores\fair_scores.ndjson"

In [22]:
combine_ndjson_files([doi_fair_scores_path, emdb_fair_scores_path], fair_scores_path)

Lines processed: 49,000,000
Finished! Total entries saved: 49,061,167


## Topics Preprocessing

### Enhance fair scores from OpenAlex with subfiled, field, and domain

In [20]:
input_topics = r"D:\pipeline-data\topics\topics_oa\topics_only_oa.ndjson"
mapping_file = r"D:\pipeline-data\external\openalex-topics\openalex_topic_mapping_table.csv"
output_topics = r"D:\pipeline-data\topics\topics_oa\topics_oa.ndjson"

In [10]:
enhance_topics(input_topics, mapping_file, output_topics)

--- Starting Line-by-Line Enhancement ---
Loading mapping CSV...
Mapping loaded. 4,516 topics indexed.
Sample Key: 'T10001'
Processing lines...
Lines processed: 15,300,000 | Matches: 15,300,000

--- Done! ---
Total Lines: 15,324,819
Total Matches: 15,324,819
Saved to: D:\pipeline-data\topics\topics_enhanced.ndjson


### Standardize and group our topics assignment from our custom model

In [7]:
input_ndjson_path = r"D:\pipeline-data\topics\topics_custom_model\topics_files"
output_path = r"D:\pipeline-data\topics\topics_custom_model\topics_custom_model.ndjson"

In [8]:
restructure_topics_ndjson(input_ndjson_path, output_path)

Processing: 772 out of 772 files...
Processing complete
Total files processed: 772
Total lines in output: 49,061,167


## Load Data to DuckDB

### Dataset metadata

#### Create metadata njson files so easier to load in table

In [3]:
slim_folder = r"D:\pipeline-data\records\slim-records"
dst_folder = r"D:\pipeline-data\dataset_index\metadata-records"

In [4]:
batch_process_metadata_from_slim(slim_folder, dst_folder)

Processing 492 files using 4 cores...
Input: D:\pipeline-data\records\slim-records
Output: D:\pipeline-data\dataset_index\metadata-records
[492/492] files completed
Done. files=492 kept=49,061,167 bad=0 time=564.7s rateâ‰ˆ86,879/rec-per-sec


{'files_seen': 492,
 'records_read': 49061167,
 'records_kept': 49061167,
 'records_bad_json': 0,
 'output_dir': 'D:\\pipeline-data\\dataset_index\\metadata-records',
 'elapsed_sec': 564.7,
 'rate_rec_per_sec': 86879}

#### Load metadata in duckdb table

In [6]:
dataset_reports_db = r"C:\Users\BPatel\Documents\dataset_index\dataset_reports.duckdb"
metadata_folder = r"D:\pipeline-data\dataset_index\metadata-records"

In [20]:
create_metadata_table(dataset_reports_db, metadata_folder)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Metadata table created in 'D:\pipeline-data\dataset_index\dataset_reports.duckdb'. Total rows: 49061167

Preview
                    dataset_id     pub_ts  pubyear  \
0  10.15156/bio/sh3578074.08fu 2021-01-01     2021   
1  10.15156/bio/sh3578075.08fu 2021-01-01     2021   
2       10.5281/zenodo.5152863 2021-01-01     2021   
3  10.15156/bio/sh3578076.08fu 2021-01-01     2021   
4  10.15156/bio/sh3578077.08fu 2021-01-01     2021   

                                            creators  \
0  [{"name":"Kõljalg, Urmas","name_type":"Persona...   
1  [{"name":"Kõljalg, Urmas","name_type":"Persona...   
2  [{"name":"Roth, Marco Pascal","identifiers":["...   
3  [{"name":"Kõljalg, Urmas","name_type":"Persona...   
4  [{"name":"Kõljalg, Urmas","name_type":"Persona...   

                                               title    source  
0                                     SH3578074.08FU  datacite  
1                                     SH3578075.08FU  datacite  
2  Seismicity catalog of hydra

In [21]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM metadata LIMIT 3").df())
con.close()

,dataset_id,pub_ts,pubyear,creators,title,source
0,10.15156/bio/sh3578074.08fu,2021-01-01,2021,"[{""name"":""Kõljalg, Urmas"",""name_type"":""Persona...",SH3578074.08FU,datacite
1,10.15156/bio/sh3578075.08fu,2021-01-01,2021,"[{""name"":""Kõljalg, Urmas"",""name_type"":""Persona...",SH3578075.08FU,datacite
2,10.5281/zenodo.5152863,2021-01-01,2021,"[{""name"":""Roth, Marco Pascal"",""identifiers"":[""...",Seismicity catalog of hydraulic-fracturing-ind...,datacite


### Citations, Mentions, FAIR scores, and Topics

In [9]:
dataset_reports_db = r"C:\Users\BPatel\Documents\dataset_index\dataset_reports.duckdb"
citations_file =  r"D:\pipeline-data\citations\citations.ndjson"
mentions_file =  r"D:\pipeline-data\mentions\mentions.ndjson"
fair_scores_file =  r"D:\pipeline-data\fair_scores\fair_scores.ndjson"

#### Load citations to duckdb

In [8]:
create_citations_table(dataset_reports_db, citations_file)

Loading Citations from: D:\pipeline-data\citations\citations.ndjson


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Citations table created. Rows: 7,669,263

Preview
            dataset_id     cit_ts  citation_year  citation_weight  \
0  10.1594/ieda/100002 2008-06-01           2008             1.00   
1  10.1594/ieda/100004 2010-01-01           2010             1.00   
2       10.5524/100001 2011-08-25           2011             1.00   
3       10.5524/100002 2012-07-02           2012             1.23   
4       10.5524/100003 2011-10-16           2011             1.00   

               source  
0             ["mdc"]  
1             ["mdc"]  
2  ["datacite","mdc"]  
3  ["mdc","openalex"]  
4  ["datacite","mdc"]  


#### Load mentions to duckdb

In [10]:
create_mentions_table(dataset_reports_db, mentions_file)

Loading Mentions from: D:\pipeline-data\mentions\mentions.ndjson
Mentions table created. Rows: 91,891

Preview
         dataset_id     men_ts  mention_year  mention_weight  source
0  10.57967/hf/0737 2023-04-04          2023             1.0  ["hf"]
1  10.57967/hf/0737 2023-05-31          2023             1.0  ["hf"]
2  10.57967/hf/0737 2023-06-08          2023             1.0  ["hf"]
3  10.57967/hf/0737 2023-06-08          2023             1.0  ["hf"]
4  10.57967/hf/0737 2023-06-13          2023             1.0  ["hf"]


#### Load FAIR scores to duckdb

In [13]:
create_fair_scores_table(dataset_reports_db, fair_scores_file)

Loading FAIR Scores from: D:\pipeline-data\fair_scores\fair_scores.ndjson


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FAIR Scores table created. Rows: 49,061,167

Preview
        dataset_id  score softwareVersion
0  10.5284/1000389  30.77           3.5.0
1  10.5284/1000140  30.77           3.5.0
2  10.5284/1000146  50.00           3.5.1
3  10.5284/1000144  30.77           3.5.0
4  10.5284/1000181  30.77           3.5.0


#### Load topics to duckdb

##### Load OpenAlex topics table

In [ ]:
oa_file = r"D:\pipeline-data\topics\topics_oa\topics_oa.ndjson"
con = duckdb.connect(dataset_reports_db)
con.execute(f"""
        CREATE OR REPLACE TABLE topics_oa AS 
        SELECT * FROM read_json_auto('{oa_file}', ignore_errors=true);
    """)
display(con.execute("SELECT * FROM topics_oa LIMIT 3").df())
con.close()

##### Load custom model topics table

In [16]:
custom_file = r"D:\pipeline-data\topics\topics_custom_model\topics_custom_model.ndjson"
con = duckdb.connect(dataset_reports_db)
con.execute(f"""
        CREATE OR REPLACE TABLE topics_custom_model AS 
        SELECT * FROM read_json_auto('{custom_file}', ignore_errors=true);
    """)
display(con.execute("SELECT * FROM topics_custom_model LIMIT 3").df())
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,dataset_id,topic_id,topic_name,score,source,subfield_id,subfield_name,field_id,field_name,domain_id,domain_name,keywords,summary,wikipedia_url
0,10.5284/1000389,T13714,Medieval Architecture and Archaeology,0.5112,custom_model,1204,Archeology,12,Arts and Humanities,2,Social Sciences,None,None,None
1,10.5284/1000140,T10087,Archaeology and ancient environmental studies,0.5078,custom_model,1911,Paleontology,19,Earth and Planetary Sciences,3,Physical Sciences,None,None,None
2,10.5284/1000146,T10889,Soil erosion and sediment transport,0.3552,custom_model,1111,Soil Science,11,Agricultural and Biological Sciences,1,Life Sciences,None,None,None


In [21]:
con = duckdb.connect(dataset_reports_db)
query = """
SELECT 
    count(*) FILTER (WHERE topic_id IS NULL) AS missing_count,
    count(*) AS total_rows
FROM topics_custom_model;
"""
print("--- Missing Values Count ---")
print(con.execute(query).df())

# 2. Previewing the rows
preview_query = """
SELECT * FROM topics_custom_model
WHERE topic_id IS NULL
LIMIT 5;
"""
display(con.execute(preview_query).df())

con.close()

--- Missing Values Count ---
   missing_count  total_rows
0          25561    49061167


,dataset_id,topic_id,topic_name,score,source,subfield_id,subfield_name,field_id,field_name,domain_id,domain_name,keywords,summary,wikipedia_url
0,10.5063/aa/pstango.3.1,None,Unclassified,0.0,custom_model,None,Unclassified,None,Unclassified,None,Unclassified,None,None,None
1,10.5063/aa/wliao.103.1,None,Unclassified,0.0,custom_model,None,Unclassified,None,Unclassified,None,Unclassified,None,None,None
2,10.5063/aa/wliao.103.2,None,Unclassified,0.0,custom_model,None,Unclassified,None,Unclassified,None,Unclassified,None,None,None
3,10.14457/psu.res.2009.5,None,Unclassified,0.0,custom_model,None,Unclassified,None,Unclassified,None,Unclassified,None,None,None
4,10.14457/kku.res.2011.3,None,Unclassified,0.0,custom_model,None,Unclassified,None,Unclassified,None,Unclassified,None,None,None


##### Create final topics table (OpenAlex if exist and score >0.5 else custom if score> than OA or OA not exist)

In [17]:
create_topics_table(dataset_reports_db)

Creating final 'topics' table with score comparison logic


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Table created in 145.33 seconds.
Final 'topics' table contains 49,061,167 rows.

Sample of rows where Custom Model won:
               dataset_id        source   score
0  10.5287/bodleian6zr1.2  custom_model  0.2431
1  10.5287/bodleian8irf.2  custom_model  0.2526
2  10.5287/bodleian2pyz.2  custom_model  0.2824
3  10.5287/bodleian6ktj.2  custom_model  0.2702
4  10.5287/bodleiandmdw.2  custom_model  0.2264


In [20]:
# Export topics for our cloud database
con = duckdb.connect(dataset_reports_db)
output_file =  r"D:\pipeline-data\topics\topics.ndjson"
con.execute(f"""
    COPY topics 
    TO '{output_file}' 
    (FORMAT JSON, ARRAY FALSE);
""")
con.close()

with open(output_file, 'rb') as f:
    line_count = sum(1 for line in f)

print(f"Export complete: {output_file}")
print(f"Total lines in file: {line_count}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Export complete: D:\pipeline-data\topics\topics_custom_model\topics.ndjson
Total lines in file: 49061167


In [22]:
def split_ndjson(input_file, target_folder, lines_per_file=500000):
    if not os.path.exists(target_folder):
        os.makedirs(target_folder)
    
    file_idx = 1
    line_count = 0
    out_file = None

    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            for line in f:
                if line_count % lines_per_file == 0:
                    if out_file:
                        out_file.close()
                    
                    chunk_path = os.path.join(target_folder, f'topics_part_{file_idx}.ndjson')
                    out_file = open(chunk_path, 'w', encoding='utf-8')
                    file_idx += 1
                
                out_file.write(line)
                line_count += 1
    finally:
        if out_file:
            out_file.close()

    print(f"Finished! Split {line_count} lines into {file_idx - 1} files.")

topics_file =  r"D:\pipeline-data\topics\topics.ndjson"
topics_split_folder =  r"D:\pipeline-data\topics\topics_split"
split_ndjson(topics_file, topics_split_folder)

Finished! Split 49061167 lines into 99 files.


In [18]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM topics WHERE source='openalex' LIMIT 3").df())
con.close()

,dataset_id,topic_id,topic_name,score,source,subfield_id,subfield_name,field_id,field_name,domain_id,domain_name
0,10.5287/bodleiandi89.2,T11150,Endoplasmic Reticulum Stress and Disease,0.734048,openalex,1307,Cell Biology,13,"Biochemistry, Genetics and Molecular Biology",1,Life Sciences
1,10.5287/bodleiannfy.2,T10513,Natural Fiber Reinforced Composites,0.253378,openalex,2507,Polymers and Plastics,25,Materials Science,3,Physical Sciences
2,10.5287/bodleian8otp.2,T10521,RNA and protein synthesis mechanisms,0.236322,openalex,1312,Molecular Biology,13,"Biochemistry, Genetics and Molecular Biology",1,Life Sciences


## Create Master Dataset Metrics table (regroup everything)

In [11]:
dataset_reports_db = r"C:\Users\BPatel\Documents\dataset_index\dataset_reports.duckdb"

In [12]:
create_dataset_metrics_table(dataset_reports_db)

Creating dataset_metrics table (topic, creators, FAIR score, 3-year metrics, etc. for each dataset)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query finished. Starting Checkpoint (saving to disk)
Checkpoint finished.
Success! dataset_metrics table created. Total datasets: 49,061,167

Preview
                  dataset_id  \
0  10.57451/lhd.ech.102357.1   
1  10.57451/lhd.ech.102358.1   
2  10.57451/lhd.ech.102359.1   
3  10.57451/lhd.ech.102360.1   
4  10.57451/lhd.ech.102361.1   

                                            creators  \
0  [{"name":"MIZUNO, Yoshinori","name_type":"Pers...   
1  [{"name":"MIZUNO, Yoshinori","name_type":"Pers...   
2  [{"name":"MIZUNO, Yoshinori","name_type":"Pers...   
3  [{"name":"MIZUNO, Yoshinori","name_type":"Pers...   
4  [{"name":"MIZUNO, Yoshinori","name_type":"Pers...   

                                 topic_name  cit_3yr  
0         Fusion and Plasma Physics Studies        0  
1         Fusion and Plasma Physics Studies        0  
2         Fusion and Plasma Physics Studies        0  
3  High-Energy Particle Collisions Research        0  
4         Fusion and Plasma Physics Studies  

In [25]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM dataset_metrics LIMIT 3").df())
con.close()

,dataset_id,pubyear,creators,dataset_source,topic_id,topic_name,topic_score,subfield_id,subfield_name,field_id,...,total_citations,total_cit_weight,cit_3yr,cit_weight_3yr,total_mentions,total_men_weight,men_3yr,men_weight_3yr,raw_dataset_index,raw_dataset_index_3yr
0,10.57451/lhd.ichvolt.88299.1,2024,"[{""name"":""SEKI, Tetsuo"",""name_type"":""Personal""...",datacite,T13769,Fusion and Plasma Physics Studies,0.4885,3109,Statistical and Nonlinear Physics,31,...,0,0.0,0,0.0,0,0.0,0,0.0,0.044867,0.044867
1,10.57451/lhd.ichvolt.88300.1,2024,"[{""name"":""SEKI, Tetsuo"",""name_type"":""Personal""...",datacite,T13769,Fusion and Plasma Physics Studies,0.4848,3109,Statistical and Nonlinear Physics,31,...,0,0.0,0,0.0,0,0.0,0,0.0,0.044867,0.044867
2,10.57451/lhd.ichvolt.88301.1,2024,"[{""name"":""SEKI, Tetsuo"",""name_type"":""Personal""...",datacite,T13769,Fusion and Plasma Physics Studies,0.4904,3109,Statistical and Nonlinear Physics,31,...,0,0.0,0,0.0,0,0.0,0,0.0,0.044867,0.044867


## Normalization factors

In [5]:
dataset_reports_db = r"C:\Users\BPatel\Documents\dataset_index\dataset_reports.duckdb"

### Create normalization factors by topics table

In [8]:
calculate_normalization_factors_topics(dataset_reports_db)

Creating normalization_factors_topics table


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Generating global normalization factors
Generating normalization factors for 4516 topics (all years)
Generating medians for target years and topic
Saving 344280 benchmark rows...
normalization_factors_topics table created.

Sample view (normalization factors for a topic for all years)
  topic_id                               topic_name  median_cit_weight_3yr  \
0   T10001      Geological and Geochemical Analysis                    0.0   
1   T10002        Advanced Chemical Physics Studies                    0.0   
2   T10003      Innovation and Knowledge Management                    0.0   
3   T10004        Soil Carbon and Nitrogen Dynamics                    0.0   
4   T10005  Ecology and Vegetation Dynamics Studies                    0.0   

   n_cit  
0  11889  
1    617  
2    302  
3  15081  
4   4243  


In [9]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM normalization_factors_topics WHERE n_cit > 20 ORDER BY median_cit_weight_3yr DESC LIMIT 5").df())
con.close()

,topic_id,topic_name,pubyear,median_fair_score_3yr,median_cit_weight_3yr,median_men_weight_3yr,n_fair,n_cit,n_men
0,T12838,Photovoltaic Systems and Sustainability,2024,13.46,35.800,0.0,304,205,205
1,T11475,French Urban and Social Studies,2025,15.38,21.045,0.0,350,258,258
2,T12838,Photovoltaic Systems and Sustainability,2025,13.46,12.550,0.0,357,234,234
3,T13974,"Health, Education, and Aging",2022,42.31,6.280,0.0,64,29,29
4,T13974,"Health, Education, and Aging",2020,42.31,3.640,0.0,41,26,26


### Create normalization factors by subfield table

In [4]:
calculate_normalization_factors_subfields(dataset_reports_db)

Creating normalization_factors_subfields table


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Generating Global Benchmark...
Generating benchmarks for 252 subfields...
Generating rolling medians for target years...
Saving 19228 benchmark rows...
normalization_factors_subfields table created.

-Preview (normalization factors for a subfield including all years) ---
  subfield_id                                 subfield_name  \
0        1100  General Agricultural and Biological Sciences   
1        1102                     Agronomy and Crop Science   
2        1103                    Animal Science and Zoology   
3        1104                               Aquatic Science   
4        1105  Ecology, Evolution, Behavior and Systematics   

   median_cit_weight_3yr    n_cit  
0                    0.0    43077  
1                    0.0    20766  
2                    0.0    52847  
3                    0.0   268095  
4                    0.0  1254859  


In [5]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM normalization_factors_subfields WHERE n_cit > 20 ORDER BY median_cit_weight_3yr DESC LIMIT 5").df())
con.close()

,subfield_id,subfield_name,pubyear,median_fair_score_3yr,median_cit_weight_3yr,median_men_weight_3yr,n_fair,n_cit,n_men
0,2604,Applied Mathematics,1998,13.46,1.23,0.0,123,56,56
1,2612,Numerical Analysis,2007,13.46,1.23,0.0,173,169,169
2,2612,Numerical Analysis,2008,13.46,1.23,0.0,177,170,170
3,2505,Materials Chemistry,<NA>,13.46,1.00,0.0,1085047,1085047,1085047
4,2610,Mathematical Physics,<NA>,15.38,1.00,0.0,60897,60897,60897


### Create floored normalization factors

#### Topics

In [10]:
create_floored_normalization_factors_table(
    db_path=dataset_reports_db,
    input_table="normalization_factors_topics",
    output_table="normalization_factors_topics_floored",
    id_col="topic_id",
    name_col="topic_name"
)

Creating floored table: normalization_factors_topics_floored (from normalization_factors_topics)
Configuration: Cit Floor=1.0, Men Floor=1.0, FAIR Min Base=10.0
-> Calculated Final FAIR Floor: 13.46
-> Inserting DEFAULT row into normalization_factors_topics_floored...
Success. normalization_factors_topics_floored created with 344,281 rows.



In [11]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT* FROM normalization_factors_topics_floored LIMIT 5").df())
con.close()

,topic_id,topic_name,pubyear,median_cit_weight_3yr,cit_is_floored,median_men_weight_3yr,men_is_floored,median_fair_score_3yr,fair_is_floored,n_fair,n_cit,n_men
0,NaN,NaN,<NA>,1.0,True,1.0,True,13.46,False,49061167,49061167,49061167
1,T10001,Geological and Geochemical Analysis,<NA>,1.0,True,1.0,True,15.38,False,11889,11889,11889
2,T10002,Advanced Chemical Physics Studies,<NA>,1.0,True,1.0,True,13.46,False,617,617,617
3,T10003,Innovation and Knowledge Management,<NA>,1.0,True,1.0,True,13.46,False,302,302,302
4,T10004,Soil Carbon and Nitrogen Dynamics,<NA>,1.0,True,1.0,True,15.38,False,15081,15081,15081


#### Subfield

In [6]:
create_floored_normalization_factors_table(
    db_path=dataset_reports_db,
    input_table="normalization_factors_subfields",
    output_table="normalization_factors_subfields_floored",
    id_col="subfield_id",
    name_col="subfield_name"
)

Creating floored table: normalization_factors_subfields_floored (from normalization_factors_subfields)
Configuration: Cit Floor=1.0, Men Floor=1.0, FAIR Min Base=10.0
-> Calculated Final FAIR Floor: 13.46
-> Inserting DEFAULT row into normalization_factors_subfields_floored...
Success. normalization_factors_subfields_floored created with 19,229 rows.



In [7]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT* FROM normalization_factors_subfields_floored LIMIT 5").df())
con.close()

,subfield_id,subfield_name,pubyear,median_cit_weight_3yr,cit_is_floored,median_men_weight_3yr,men_is_floored,median_fair_score_3yr,fair_is_floored,n_fair,n_cit,n_men
0,NaN,NaN,<NA>,1.0,True,1.0,True,13.46,False,49061167,49061167,49061167
1,1100,General Agricultural and Biological Sciences,<NA>,1.0,True,1.0,True,13.46,False,43077,43077,43077
2,1102,Agronomy and Crop Science,<NA>,1.0,True,1.0,True,13.46,False,20766,20766,20766
3,1103,Animal Science and Zoology,<NA>,1.0,True,1.0,True,13.46,False,52847,52847,52847
4,1104,Aquatic Science,<NA>,1.0,True,1.0,True,13.46,False,268095,268095,268095


In [7]:
# save for live Dataset Index calculation
target_db = r"D:\pipeline-data\dataset_index\subfield_norm_factors.duckdb"
table_name = 'normalization_factors_subfields_floored'

con = duckdb.connect(dataset_reports_db)

try:
    con.execute(f"ATTACH '{target_db}' AS dest_db")

    con.execute(f"""
        CREATE TABLE dest_db.{table_name} 
        AS SELECT * FROM main.{table_name}
    """)
    
    print(f"Successfully saved '{table_name}' to {target_db}")

finally:
    con.close()

CatalogException: Catalog Error: Table with name "normalization_factors_subfields_floored" already exists!

## Dataset Index

In [12]:
dataset_reports_db = r"C:\Users\BPatel\Documents\dataset_index\dataset_reports.duckdb"

In [13]:
create_dataset_index_table(dataset_reports_db)

Creating dataset_index related tables
Generating numeric 'dataset_norm_index' (writing to disk)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

-Creating virtual view 'dataset_index'
Success! Process completed in 133.15 seconds.
Physical table 'dataset_norm_index': 49,061,167 rows.
Virtual View 'dataset_index' ready for analysis

 Preview of dataset_index


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                     dataset_id  pubyear  \
0  10.57451/lhd.gpcradh.78051.1     2024   
1  10.57451/lhd.gpcradh.78052.1     2024   
2  10.57451/lhd.gpcradh.78053.1     2024   
3  10.57451/lhd.gpcradh.78054.1     2024   
4  10.57451/lhd.gpcradh.78055.1     2024   

                                            creators dataset_source topic_id  \
0  [{"name":"TOKUZAWA, Tokihiko","name_type":"Per...       datacite   T10597   
1  [{"name":"TOKUZAWA, Tokihiko","name_type":"Per...       datacite   T10597   
2  [{"name":"TOKUZAWA, Tokihiko","name_type":"Per...       datacite   T10597   
3  [{"name":"TOKUZAWA, Tokihiko","name_type":"Per...       datacite   T10597   
4  [{"name":"TOKUZAWA, Tokihiko","name_type":"Per...       datacite   T10597   

                                topic_name  topic_score subfield_id  \
0  Nuclear reactor physics and engineering       0.4706        2202   
1  Nuclear reactor physics and engineering       0.4723        2202   
2  Nuclear reactor physics and engineerin

In [10]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT* FROM dataset_index WHERE dataset_index_subfield > 1 LIMIT 5").df())
con.close()

,dataset_id,pubyear,creators,dataset_source,topic_id,topic_name,topic_score,subfield_id,subfield_name,field_id,...,t_norm_men,t_norm_gap,t_source,s_norm_fair,s_norm_cit,s_norm_men,s_norm_gap,s_source,dataset_index_topic,dataset_index_subfield
0,10.3535/tfe-jwy-8e9,2025,"[{""name"":""Naturalis Biodiversity Center"",""name...",datacite,T10992,Forensic Anthropology and Bioarchaeology Studies,0.3831,1204,Archeology,12,...,1.0,0,Exact Year,13.46,1.0,1.0,0,Exact Year,1.381129,1.381129
1,10.3535/lp0-rjs-q0y,2025,"[{""name"":""Naturalis Biodiversity Center"",""name...",datacite,T13015,"Botany, Ecology, and Taxonomy Studies",0.4474,1110,Plant Science,11,...,1.0,0,Exact Year,13.46,1.0,1.0,0,Exact Year,1.381129,1.381129
2,10.34740/kaggle/dsv/12371336,2025,"[{""name"":""Andrés Chirinos""}]",datacite,T10742,Peer-to-Peer Network Technologies,0.6433,1705,Computer Networks and Communications,17,...,1.0,0,Exact Year,13.46,1.0,1.0,0,Exact Year,1.042046,1.190688
3,10.3535/ydz-dva-cfv,2025,"[{""name"":""Naturalis Biodiversity Center"",""name...",datacite,T13015,"Botany, Ecology, and Taxonomy Studies",0.4588,1110,Plant Science,11,...,1.0,0,Exact Year,13.46,1.0,1.0,0,Exact Year,1.190688,1.190688
4,10.15468/dl.nx7344,2025,"[{""name"":""GBIF.org User"",""name_type"":""Organiza...",datacite,T14435,Information Retrieval and Data Mining,0.3168,1710,Information Systems,17,...,1.0,0,Exact Year,13.46,1.0,1.0,0,Exact Year,1.190688,1.190688


## S-index

In [3]:
dataset_reports_db = r"C:\Users\BPatel\Documents\dataset_index\dataset_reports.duckdb"
temp_dir = r"C:\Users\BPatel\Documents\dataset_index\temp"

### Create a creators_table first exploding the dataset_index table on creators first

In [15]:
create_creators_table(dataset_reports_db, temp_dir = temp_dir)

FULL RUN: Creating optimized creators table
-> Setting temp directory to: C:\Users\BPatel\Documents\dataset_index\temp
Exploding and parsing creators (writing minimal data to disk)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Creating virtual view 'creators_table'
Success! Process completed in 1614.55 seconds.
Physical Table 'dataset_creators_parsed': 216,688,512 rows.

Preview:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

       creator_name   primary_identifier
0  NAGAYAMA, Yoshio  0000-0001-8887-7541
1  NAGAYAMA, Yoshio  0000-0001-8887-7541
2  NAGAYAMA, Yoshio  0000-0001-8887-7541
3  NAGAYAMA, Yoshio  0000-0001-8887-7541
4  NAGAYAMA, Yoshio  0000-0001-8887-7541


In [ ]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM creators_table ORDER BY total_cit_weight DESC LIMIT 5").df())
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### Create S-index table by matching identifier first then name/affiliation

In [4]:
create_s_index_identifier_name_affiliation_table(dataset_reports_db, temp_dir=temp_dir)

Creating s_index_identifier_name_affiliation table
Executing Single-Pass Aggregation...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Success! Completed in 1367.61 seconds.
Total authors: 1,445,697


In [5]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT* from s_index_identifier_name_affiliation LIMIT 5").df())
con.close()

,distinct_group_id,grouping_method,display_name,primary_identifier,all_affiliations,name_type,primary_topic_id,primary_topic_name,primary_subfield_id,primary_subfield_name,...,n_datasets,S_index_topics,S_index_subfield,avg_dataset_index_topics,avg_dataset_index_subfield,total_cit_weight,total_men_weight,sum_total_citations,sum_total_mentions,avg_fair_score
0,NaN,name_affiliation,GBIF.Org User,NaN,[],Personal,T10015,Genomics and Phylogenetic Studies,1312,Molecular Biology,...,116357217,5.230931e+07,6.244446e+07,0.449558,0.536662,13499970.81,181800.95,11480781.0,121128.0,27.654848
1,0000-0001-5473-2109,identifier,"TOKUZAWA, Tokihiko",0000-0001-5473-2109,"[National Institute for Fusion Science (NIFS),...",Personal,T13769,Fusion and Plasma Physics Studies,3109,Statistical and Nonlinear Physics,...,5727142,1.239588e+06,1.516425e+06,0.216441,0.264779,0.00,0.00,0.0,0.0,14.673781
2,https://nrid.nii.ac.jp/nrid/1000050260047/,identifier,"TANAKA, Kenji",https://nrid.nii.ac.jp/nrid/1000050260047/,"[National Institute for Fusion Science (NIFS),...",Personal,T13769,Fusion and Plasma Physics Studies,3109,Statistical and Nonlinear Physics,...,3208893,9.879175e+05,9.844533e+05,0.307869,0.306789,0.00,0.00,0.0,0.0,14.785334
3,https://ror.org/0566bfb96,identifier,Naturalis Biodiversity Center,https://ror.org/0566bfb96,[],Organizational,T11974,Lepidoptera: Biology and Taxonomy,1311,Genetics,...,2388894,9.768002e+05,9.878013e+05,0.408892,0.413497,1.00,0.00,1.0,0.0,17.094411
4,https://nrid.nii.ac.jp/nrid/1000040300727/,identifier,"FUNABA, Hisamichi",https://nrid.nii.ac.jp/nrid/1000040300727/,"[National Institute for Fusion Science (NIFS),...",Personal,T13769,Fusion and Plasma Physics Studies,3109,Statistical and Nonlinear Physics,...,2104401,8.062265e+05,7.891316e+05,0.383114,0.374991,242317.38,0.00,197006.0,0.0,15.511266


In [6]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT* from s_index_identifier_name_affiliation WHERE name_type='Organizational' LIMIT 10").df())
con.close()

,distinct_group_id,grouping_method,display_name,primary_identifier,all_affiliations,name_type,primary_topic_id,primary_topic_name,primary_subfield_id,primary_subfield_name,...,n_datasets,S_index_topics,S_index_subfield,avg_dataset_index_topics,avg_dataset_index_subfield,total_cit_weight,total_men_weight,sum_total_citations,sum_total_mentions,avg_fair_score
0,https://ror.org/0566bfb96,identifier,Naturalis Biodiversity Center,https://ror.org/0566bfb96,[],Organizational,T11974,Lepidoptera: Biology and Taxonomy,1311,Genetics,...,2388894,976800.235498,987801.289011,0.408892,0.413497,1.00,0.00,1.0,0.0,17.094411
1,https://ror.org/01faaaf77,identifier,University of Graz,https://ror.org/01faaaf77,[],Organizational,T12618,Botany and Plant Ecology Studies,1110,Plant Science,...,533246,220460.489439,230140.657584,0.413431,0.431584,1.00,0.00,1.0,0.0,17.696206
2,https://ror.org/01tv5y993,identifier,Natural History Museum Vienna,https://ror.org/01tv5y993,[],Organizational,T14120,Thallium and Germanium Studies,2728,Neurology,...,377634,184679.267102,193348.242949,0.489043,0.511999,0.00,0.00,0.0,0.0,22.024863
3,https://ror.org/00bv4cx53,identifier,Botanic Garden and Botanical Museum Berlin,https://ror.org/00bv4cx53,[],Organizational,T12618,Botany and Plant Ecology Studies,1110,Plant Science,...,414555,170581.692771,172931.136519,0.411481,0.417149,0.00,0.00,0.0,0.0,17.195126
4,https://ror.org/0443cwa12,identifier,Tallinn University of Technology,https://ror.org/0443cwa12,[],Organizational,T13021,Geology and Environmental Impact Studies,3308,Law,...,263312,131029.682916,133526.399476,0.497621,0.507103,0.00,0.00,0.0,0.0,20.783665
5,https://ror.org/040ck2b86,identifier,Natural History Museum of Denmark,https://ror.org/040ck2b86,[],Organizational,T11271,Myasthenia Gravis and Thymoma,2713,Epidemiology,...,74185,27214.970007,29271.452196,0.366853,0.394574,0.00,0.00,0.0,0.0,16.852110
6,https://ror.org/01wz97s39,identifier,Senckenberg Research Institute and Natural His...,https://ror.org/01wz97s39,[],Organizational,T14120,Thallium and Germanium Studies,1311,Genetics,...,52530,19781.933014,20665.256718,0.376584,0.393399,0.00,0.00,0.0,0.0,16.788003
7,11733776,identifier,CXC-DS,11733776,[],Organizational,T10744,Astrophysical Phenomena and Observations,3103,Astronomy and Astrophysics,...,39408,12672.942356,13466.115177,0.321583,0.341710,285.40,1.36,267.0,1.0,15.891892
8,https://ror.org/039zvsn29,identifier,Natural History Museum,https://ror.org/039zvsn29,"[Paul Jenkins, Peter James, C. J. O. Harrison,...",Organizational,T11462,Museums and Cultural Heritage,1209,Museology,...,9,11775.846115,11776.107724,1308.427346,1308.456414,35266.92,11.95,21544.0,7.0,74.146667
9,"master, daniel m._[wheaton college]",name_affiliation,"Master, Daniel M.",NaN,[Wheaton College],Organizational,T12370,Advances in Cucurbitaceae Research,1311,Genetics,...,28218,10017.836650,10317.713847,0.355016,0.365643,0.00,0.00,0.0,0.0,15.277155


#### View table excluding name_type='Organizational' and display_name='GBIF.Org User'

In [7]:
con = duckdb.connect(dataset_reports_db)
con.execute("""
    CREATE OR REPLACE VIEW s_index_clean AS
    SELECT *
    FROM s_index_identifier_name_affiliation
    WHERE NOT (
        name_type = 'Organizational' 
        OR display_name = 'GBIF.Org User'
    )
""")
con.close()

In [8]:
con = duckdb.connect(dataset_reports_db)
result_df = con.execute("SELECT COUNT(*) AS total_authors FROM s_index_clean").df()
total_authors = result_df['total_authors'].iloc[0]
print(f"Total Authors after removing Organizational authors: {total_authors:,}")
con.close()

Total Authors after removing Organizational authors: 1,032,543


## ---- NOT NEEDED ----

In [13]:
con = duckdb.connect(dataset_reports_db)
query = """
SELECT 
    (SELECT COUNT(*) FROM dataset_metrics) as dataset_metrics,
    (SELECT COUNT(*) FROM dataset_norm_index) as dataset_norm_index,
    (SELECT COUNT(*) FROM creators_table) as creators_table,
    (SELECT COUNT(*) FROM dataset_creators_parsed) as dataset_creators_parsed
    """
display(con.execute(query).df())
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,dataset_metrics,dataset_norm_index,creators_table,dataset_creators_parsed
0,49061167,49061167,216688512,216688512


In [15]:
con = duckdb.connect(dataset_reports_db)
query = """
SELECT primary_identifier, COUNT(*) 
FROM dataset_creators_parsed
WHERE len(primary_identifier) < 20 -- Look for clean, short IDs
AND primary_identifier NOT LIKE 'http%' -- That are fully cleaned
GROUP BY 1
ORDER BY 2 DESC
LIMIT 5;
    """
display(con.execute(query).df())
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,primary_identifier,count_star()
0,0000-0001-5473-2109,5727142
1,0000-0002-9160-682x,2608151
2,0000-0001-5220-947x,2557077
3,0000-0002-5892-6047,2014769
4,0000-0003-0161-0938,1952825


In [16]:
con = duckdb.connect(dataset_reports_db)
query = """
SELECT sum(n_datasets) FROM s_index_identifier_name_affiliation;
    """
display(con.execute(query).df())
con.close()

,sum(n_datasets)
0,216513651.0


In [19]:
con = duckdb.connect(dataset_reports_db)
query = """
    SELECT COUNT(*) as ghost_authors
    FROM dataset_creators_parsed
    WHERE 
        (creator_name IS NULL OR TRIM(creator_name) = '')
        AND 
        (primary_identifier IS NULL OR TRIM(primary_identifier) = '')
"""
display(con.execute(query).df())
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ghost_authors
0,175293


In [20]:
with duckdb.connect(dataset_reports_db) as con:
    print(
        con.execute("""
        SELECT 
            distinct_group_id, 
            display_name, 
            primary_identifier, 
            all_affiliations, 
            n_datasets
        FROM s_index_identifier_name_affiliation
        WHERE 
            -- Catch rows where name is technically present but empty
            TRIM(display_name) = '' 
            AND primary_identifier IS NULL
        LIMIT 10
        """).df()
    )

Empty DataFrame
Columns: [distinct_group_id, display_name, primary_identifier, all_affiliations, n_datasets]
Index: []


In [21]:
with duckdb.connect(dataset_reports_db) as con:
    print(
        con.execute("""
        SELECT 
            distinct_group_id, 
            display_name, 
            primary_identifier, 
            n_datasets
        FROM s_index_identifier_name_affiliation
        WHERE 
            TRIM(display_name) = '' 
            AND 
            (primary_identifier IS NULL OR TRIM(primary_identifier) = '')
        LIMIT 10
        """).df()
    )

Empty DataFrame
Columns: [distinct_group_id, display_name, primary_identifier, n_datasets]
Index: []


### Create S-index table by matching identifiers only

In [79]:
create_s_index_identifier_table(dataset_reports_db)

Initializing S_index_identifier (Master Researcher Profile)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--------------------------------------------------
Success! S_index_identifier created.
Total Unique Researchers based on identifier: 441,944
Execution Time: 358.95 seconds
--------------------------------------------------

Top 5 Researchers (with Topic/Subfield IDs):
                           primary_identifier  \
0                         0000-0001-5473-2109   
1  https://nrid.nii.ac.jp/nrid/1000050260047/   
2                   https://ror.org/0566bfb96   
3                         0000-0002-9160-682x   
4  https://nrid.nii.ac.jp/nrid/1000040300727/   

                     primary_topic_name            primary_subfield_name  \
0     Geochemistry and Geologic Mapping          Artificial Intelligence   
1  Magnetic confinement fusion research  Nuclear and High Energy Physics   
2     Geochemistry and Geologic Mapping          Artificial Intelligence   
3    Prenatal Screening and Diagnostics                Molecular Biology   
4  Magnetic confinement fusion research  Nuclear and Hi

In [113]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM S_index_identifier LIMIT 5").df())
con.close()

,primary_identifier,creator_names,name_type,all_affiliations,primary_topic_id,primary_topic_name,primary_subfield_id,primary_subfield_name,n_unique_topics,n_unique_subfields,...,n_datasets,S_index_topics,S_index_subfield,avg_dataset_index_topics,avg_dataset_index_subfield,total_cit_weight,total_men_weight,sum_total_citations,sum_total_mentions,avg_fair_score
0,0000-0001-5473-2109,"[TOKUZAWA, Tokihiko]",Personal,"[National Institute for Fusion Science, Nation...",T12157,Geochemistry and Geologic Mapping,1702,Artificial Intelligence,4499,252,...,5727142,1.393599e+06,1.400136e+06,0.243332,0.244474,0.00,0.00,0.0,0.0,14.673781
1,https://nrid.nii.ac.jp/nrid/1000050260047/,"[TANAKA, Kenji]",Personal,"[National Institute for Fusion Science, Nation...",T10346,Magnetic confinement fusion research,3106,Nuclear and High Energy Physics,4491,252,...,3208893,7.884570e+05,7.903823e+05,0.245710,0.246310,0.00,0.00,0.0,0.0,14.785334
2,https://ror.org/0566bfb96,"[Naturalis Biodiversity Center, Distributed Sy...",Organizational,[],T12157,Geochemistry and Geologic Mapping,1702,Artificial Intelligence,4255,251,...,2388894,6.805641e+05,6.805869e+05,0.284887,0.284896,1.13,1.13,1.0,1.0,17.094411
3,0000-0002-9160-682x,"[GOTO, Motoshi]",Personal,"[National Institute for Fusion Science (NIFS),...",T10978,Prenatal Screening and Diagnostics,1312,Molecular Biology,4504,252,...,2608151,6.344953e+05,6.364951e+05,0.243274,0.244041,0.00,0.00,0.0,0.0,14.659584
4,https://nrid.nii.ac.jp/nrid/1000040300727/,"[FUNABA, Hisamichi]",Personal,"[National Institute for Fusion Science (NIFS),...",T10346,Magnetic confinement fusion research,3106,Nuclear and High Energy Physics,4490,252,...,2104401,6.296659e+05,6.357325e+05,0.299214,0.302097,276241.20,0.00,197006.0,0.0,15.511266


### Create S-index table by matching name and affiliations set

In [5]:
create_s_index_name_affiliation_table(dataset_reports_db)

Creating s_index_name_affiliation table


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Success! 's_index_name_affiliation' created.
Total Unique Name-Affiliation Sets: 3,962,943
Execution Time: 289.25 seconds

Preview Top 5 Rows:
          grouping_name affiliation_set_signature  n_datasets  S_index_topics
0    nilsson, r. henrik                      None     3201095    1.619952e+06
1      abarenkov, kessy                      None     3201073    1.619924e+06
2        kõljalg, urmas                      None     3201065    1.619920e+06
3  larsson, karl-henrik                      None     3200948    1.619337e+06
4        tedersoo, leho                      None     2383786    1.224757e+06


In [6]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM S_index_name_affiliation LIMIT 5").df())
con.close()

,grouping_name,affiliation_set_signature,name_type,primary_topic_id,primary_topic_name,primary_subfield_id,primary_subfield_name,n_unique_topics,n_unique_subfields,first_pub_year,...,n_datasets,S_index_topics,S_index_subfield,avg_dataset_index_topics,avg_dataset_index_subfield,total_cit_weight,total_men_weight,sum_total_citations,sum_total_mentions,avg_fair_score
0,"nilsson, r. henrik",None,Personal,T10451,Mycorrhizal Fungi and Plant Interactions,1110,Plant Science,4477,252,2012,...,3201095,1.619952e+06,1.627984e+06,0.506062,0.508571,864.65,814.86,625.0,590.0,31.049449
1,"abarenkov, kessy",None,Personal,T10451,Mycorrhizal Fungi and Plant Interactions,1110,Plant Science,4477,252,2012,...,3201073,1.619924e+06,1.627956e+06,0.506056,0.508566,862.65,812.86,623.0,588.0,31.049163
2,"kõljalg, urmas",None,Personal,T10451,Mycorrhizal Fungi and Plant Interactions,1110,Plant Science,4477,252,2015,...,3201065,1.619920e+06,1.627953e+06,0.506057,0.508566,864.87,815.08,625.0,590.0,31.049142
3,"larsson, karl-henrik",None,Personal,T10451,Mycorrhizal Fungi and Plant Interactions,1110,Plant Science,4477,252,2015,...,3200948,1.619337e+06,1.627369e+06,0.505893,0.508402,57.90,57.90,43.0,43.0,31.049087
4,"tedersoo, leho",None,Personal,T10451,Mycorrhizal Fungi and Plant Interactions,1110,Plant Science,4471,252,2015,...,2383786,1.224757e+06,1.228910e+06,0.513787,0.515529,2009.15,69.15,1992.0,53.0,30.919935
